# 02 — MAESTRO post-training analysis

This notebook demonstrates downstream analyses you can perform after training a MAESTRO model on CyTOF data. It covers:

1. **Loading a trained model and data**
2. **Reconstruction visualization** — masking cells and comparing reconstructions
3. **Latent embedding extraction** — generating patient-level representations
4. **UMAP visualization** — embedding space colored by metadata (diagnosis, sex, age)
5. **Diagnosis classification** — logistic regression on latent embeddings
6. **Sex prediction** — binary classification from embeddings
7. **Age regression** — Ridge regression from embeddings
8. **Cell type proportion prediction** — predicting gated proportions from embeddings
9. **Batch effect analysis** — measuring robustness to marker intensity shifts
10. **Pool attention interpretation** — understanding which cell types the model attends to

**Note:** This notebook uses the toy/demo data included in this repository. Paths are relative to the repo root. Adjust paths and hyperparameters for your own data.

## 0. Setup & Imports

In [ ]:
import sys
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import umap
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    mean_absolute_error,
    r2_score,
)
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from tqdm.notebook import tqdm

# Support launching Jupyter from either the repository root or notebooks/.
project_root = Path.cwd().resolve()
if not (project_root / "src").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
from src.data.dataset import CyTOFDataset
from src.models.maestro import MAESTRO, ruler_masking

warnings.filterwarnings("ignore")

plt.style.use("default")
plt.rcParams.update(
    {
        "figure.figsize": (5, 4),
        "font.family": "sans-serif",
        "figure.dpi": 150,
        "axes.grid": False,
    }
)

print("Imports complete.")

## 1. Load Data & Model

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

### 1a. Load the CyTOF dataset

The demo data contains two panels (dataA and dataB) with different markers. `CyTOFDataset` automatically discovers the shared markers across panels.

In [ ]:
demo_data_directory = project_root / "data/h5/dataA"
dataset = CyTOFDataset(data_dirs=[demo_data_directory])

print(f"Number of samples: {len(dataset)}")
print(f"Shared markers ({len(dataset.shared_markers)}): {dataset.shared_markers}")

In [ ]:
sample_tensors = []
sample_cell_types = []
sample_names = []

for sample_index in tqdm(range(len(dataset)), desc="Loading samples"):
    sample_tensor, cell_types, sample_name = dataset[sample_index]
    sample_tensors.append(sample_tensor.unsqueeze(0))
    sample_cell_types.append(cell_types)
    sample_names.append(sample_name)

print(f"Loaded {len(sample_tensors)} samples.")
print(f"Example sample shape: {sample_tensors[0].shape}")
print(f"Discovered cell types: {dataset.get_all_cell_types()}")

### 1b. Load metadata

In [ ]:
metadata = pd.read_csv(project_root / "data/csv/metadata.csv")

# Convert source paths such as dataA/patient_001.csv to dataset sample names.
metadata["sample_name"] = metadata["file"].apply(lambda file_path: Path(file_path).stem)

print(f"Metadata columns: {list(metadata.columns)}")
print(f"Diagnoses: {metadata['Diagnosis'].unique()}")
metadata.head()

### 1c. Load a trained MAESTRO model

The upstream ToyModel path is selected below. Because `output/` is local and ignored, place the checkpoint there or point these paths at an experiment directory.

In [ ]:
MODEL_CHECKPOINT_PATH = project_root / "output/training/ToyModel/model.ckpt"
MODEL_CONFIGURATION_PATH = project_root / "output/training/ToyModel/config.pth"

if not MODEL_CONFIGURATION_PATH.exists():
    print(
        "WARNING: Model config not found. Please update "
        "MODEL_CHECKPOINT_PATH and MODEL_CONFIGURATION_PATH above."
    )
    print(
        "Skipping model loading. Downstream cells that require the model will not work."
    )
    model = None
else:
    model_configuration = torch.load(MODEL_CONFIGURATION_PATH, map_location=device)
    print("Model config:", dict(model_configuration))

    # Load checkpoint — handles both Lightning and raw state_dict formats
    checkpoint = torch.load(MODEL_CHECKPOINT_PATH, map_location=device)
    if "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
        state_dict = {
            state_key.replace("model.", "", 1): state_value
            for state_key, state_value in state_dict.items()
        }
    else:
        state_dict = checkpoint
    # The packaged implementation uses a descriptive name for this buffer.
    if "centerlatent" in state_dict:
        state_dict["center_latent"] = state_dict.pop("centerlatent")

    model = MAESTRO(
        dim_input=model_configuration["dim_input"],
        dim_output=model_configuration["dim_output"],
        num_inds=model_configuration["num_inds"],
        dim_hidden=model_configuration["dim_hidden"],
        dim_latent=model_configuration["dim_latent"],
        num_heads=model_configuration["num_heads"],
        num_outputs=model_configuration["num_outputs"],
        ln=model_configuration["ln"],
        number_cells_subset=model_configuration["number_cells_subset"],
        student_temperature=model_configuration["student_temperature"],
        teacher_temperature=model_configuration["teacher_temperature"],
        teacher_temperature_step=model_configuration.get(
            "teacher_temperature_step", 0.01
        ),
        sinkhorn_start=model_configuration.get("sinkhorn_start", 0),
        center_momentum=model_configuration.get("center_momentum", 0.9),
        teacher_beta=model_configuration.get("teacher_beta", 0.99),
    ).to(device)

    model.load_state_dict(state_dict, strict=True)
    model.eval()
    print("Model loaded successfully.")

---
## 2. Reconstruction Visualization

Mask a portion of cells and compare the model's reconstruction to the original data using UMAP.

In [ ]:
cell_type_colors = {
    "Neutrophil": "#0F4C81",
    "MemoryCD4T": "#FF6F61",
    "NaiveCD4T": "#009874",
    "MemoryCD8T": "#F5DF4D",
    "NaiveCD8T": "#BF1932",
    "Bcell": "#92A8D1",
    "ClassicalMono": "#88B04B",
    "NonClassicalMono": "#9B1B30",
    "NKcell": "#E2583E",
    "Treg": "#5A5B9F",
    "mDC": "#F0C05A",
    "pDC": "#53B0AE",
    "Eosinophil": "#F7CAC9",
    "Basophil": "#5F4B8B",
}
default_color = "#AAAAAA"

number_visualized_samples = 3
visualization_sample_indices = np.random.choice(
    len(sample_tensors), number_visualized_samples, replace=False
)
mask_ratio = 0.5

_, subplot_axes = plt.subplots(
    number_visualized_samples,
    4,
    figsize=(16, 4 * number_visualized_samples),
)
column_titles = [
    "Cell Types (original)",
    "Unmasked",
    "Masked (target)",
    "Predicted",
]

for row_index, sample_index in enumerate(visualization_sample_indices):
    with torch.no_grad():
        sample_tensor = sample_tensors[sample_index].to(device)
        cell_type_indices = sample_cell_types[sample_index]
        sample_name = sample_names[sample_index]

        number_cells = sample_tensor.shape[1]
        visualization_size = min(5000, number_cells)
        selected_cell_indices = torch.randperm(number_cells)[:visualization_size]
        sampled_input = sample_tensor[:, selected_cell_indices, :]
        sampled_cell_type_indices = cell_type_indices[selected_cell_indices]

        mask = ruler_masking(sampled_input, mask_ratio)
        batch_size, _, marker_count = sampled_input.shape
        unmasked_cells = sampled_input[~mask].view(batch_size, -1, marker_count)
        masked_target_cells = sampled_input[mask].view(batch_size, -1, marker_count)

        encoded_cells, *_ = model.student.forward_encoder(unmasked_cells)
        sample_embedding, _, _ = model.student.forward_pooling(encoded_cells)
        reconstructed_cells = model.student.forward_decoder(sample_embedding, mask)

        number_masked_cells = masked_target_cells.shape[1]
        prediction_indices = torch.randperm(reconstructed_cells.shape[1])[
            :number_masked_cells
        ]
        sampled_predictions = reconstructed_cells[:, prediction_indices, :]

        unmasked_values = unmasked_cells.squeeze(0).float().cpu().numpy()
        target_values = masked_target_cells.squeeze(0).float().cpu().numpy()
        prediction_values = sampled_predictions.squeeze(0).float().cpu().numpy()

    combined_values = np.vstack([unmasked_values, target_values, prediction_values])
    umap_reducer = umap.UMAP(n_neighbors=30, min_dist=0.3, random_state=42)
    embedding = umap_reducer.fit_transform(combined_values)

    number_unmasked = len(unmasked_values)
    number_targets = len(target_values)
    unmasked_embedding = embedding[:number_unmasked]
    target_embedding = embedding[number_unmasked : number_unmasked + number_targets]
    prediction_embedding = embedding[number_unmasked + number_targets :]

    cpu_mask = mask.squeeze(0).cpu()
    unmasked_indices = (~cpu_mask).nonzero(as_tuple=True)[0]
    masked_indices = cpu_mask.nonzero(as_tuple=True)[0]
    unmasked_cell_type_names = [
        dataset.get_cell_type_name(sampled_cell_type_indices[index].item())
        for index in unmasked_indices
    ]
    masked_cell_type_names = [
        dataset.get_cell_type_name(sampled_cell_type_indices[index].item())
        for index in masked_indices
    ]
    cell_type_names = unmasked_cell_type_names + masked_cell_type_names

    x_limits = [embedding[:, 0].min() - 0.5, embedding[:, 0].max() + 0.5]
    y_limits = [embedding[:, 1].min() - 0.5, embedding[:, 1].max() + 0.5]

    subplot_axis = subplot_axes[row_index, 0]
    point_colors = [
        cell_type_colors.get(cell_type, default_color) for cell_type in cell_type_names
    ]
    subplot_axis.scatter(
        np.vstack([unmasked_embedding, target_embedding])[:, 0],
        np.vstack([unmasked_embedding, target_embedding])[:, 1],
        c=point_colors,
        s=1,
        alpha=0.5,
    )
    subplot_axis.set_xlim(x_limits)
    subplot_axis.set_ylim(y_limits)
    subplot_axis.set_xticks([])
    subplot_axis.set_yticks([])
    subplot_axis.set_ylabel(sample_name, fontsize=9)
    if row_index == 0:
        subplot_axis.set_title(column_titles[0], fontsize=10)

    subplot_axis = subplot_axes[row_index, 1]
    subplot_axis.scatter(
        unmasked_embedding[:, 0],
        unmasked_embedding[:, 1],
        c="#FFB703",
        s=1,
        alpha=0.5,
    )
    subplot_axis.set_xlim(x_limits)
    subplot_axis.set_ylim(y_limits)
    subplot_axis.set_xticks([])
    subplot_axis.set_yticks([])
    if row_index == 0:
        subplot_axis.set_title(column_titles[1], fontsize=10)

    subplot_axis = subplot_axes[row_index, 2]
    subplot_axis.scatter(
        target_embedding[:, 0],
        target_embedding[:, 1],
        c="#E63946",
        s=1,
        alpha=0.5,
    )
    subplot_axis.set_xlim(x_limits)
    subplot_axis.set_ylim(y_limits)
    subplot_axis.set_xticks([])
    subplot_axis.set_yticks([])
    if row_index == 0:
        subplot_axis.set_title(column_titles[2], fontsize=10)

    subplot_axis = subplot_axes[row_index, 3]
    subplot_axis.scatter(
        prediction_embedding[:, 0],
        prediction_embedding[:, 1],
        c="#0077B6",
        s=1,
        alpha=0.5,
    )
    subplot_axis.set_xlim(x_limits)
    subplot_axis.set_ylim(y_limits)
    subplot_axis.set_xticks([])
    subplot_axis.set_yticks([])
    if row_index == 0:
        subplot_axis.set_title(column_titles[3], fontsize=10)

plt.suptitle(f"Reconstruction Visualization (mask ratio = {mask_ratio})", fontsize=13)
plt.tight_layout()
plt.show()

---
## 3. Latent Embedding Extraction

Generate a single latent vector per patient by passing all cells through the encoder and pooling layer.

In [ ]:
assert model is not None, "Model not loaded."

sample_embeddings = {}

for sample_index in tqdm(
    range(len(sample_tensors)), desc="Generating latent embeddings"
):
    with torch.no_grad():
        sample_tensor = sample_tensors[sample_index].to(device)
        encoded_cells, *_ = model.student.forward_encoder(sample_tensor)
        sample_embedding, _, _ = model.student.forward_pooling(encoded_cells)
        sample_embeddings[sample_names[sample_index]] = sample_embedding.cpu()

embedding_values = {
    sample_name: embedding_tensor.flatten().numpy()
    for sample_name, embedding_tensor in sample_embeddings.items()
}

embedding_dimension = next(iter(embedding_values.values())).shape[0]
embedding_frame = pd.DataFrame.from_dict(embedding_values, orient="index")
embedding_frame.reset_index(inplace=True)
embedding_frame.columns = ["sample_name"] + [
    f"Feature_{feature_index + 1}" for feature_index in range(embedding_dimension)
]

print(f"Latent DataFrame shape: {embedding_frame.shape}")
embedding_frame.head()

---
## 4. UMAP Visualization of Latent Embeddings

Visualize the patient-level latent space, colored by diagnosis, sex, and age.

In [ ]:
analysis_frame = pd.merge(embedding_frame, metadata, on="sample_name", how="inner")
print(f"Merged DataFrame shape: {analysis_frame.shape}")

feature_columns = [
    column for column in analysis_frame.columns if column.startswith("Feature_")
]
latent_matrix = analysis_frame[feature_columns].values

umap_reducer = umap.UMAP(random_state=42, n_neighbors=15, min_dist=0.5)
umap_embedding = umap_reducer.fit_transform(latent_matrix)
analysis_frame["UMAP1"] = umap_embedding[:, 0]
analysis_frame["UMAP2"] = umap_embedding[:, 1]

In [ ]:
diagnosis_colors = {
    "DiagnosisA": "#FF595E",
    "DiagnosisB": "#006DB8",
    "DiagnosisC": "#FFCA3A",
}

_, subplot_axes = plt.subplots(1, 3, figsize=(15, 4))

subplot_axis = subplot_axes[0]
for diagnosis, color in diagnosis_colors.items():
    diagnosis_mask = analysis_frame["Diagnosis"] == diagnosis
    subplot_axis.scatter(
        analysis_frame.loc[diagnosis_mask, "UMAP1"],
        analysis_frame.loc[diagnosis_mask, "UMAP2"],
        c=color,
        label=diagnosis,
        s=15,
        alpha=0.7,
    )
subplot_axis.set_title("Colored by Diagnosis")
subplot_axis.legend(fontsize=8, markerscale=2)
subplot_axis.set_xticks([])
subplot_axis.set_yticks([])
subplot_axis.set_xlabel("UMAP1")
subplot_axis.set_ylabel("UMAP2")

sex_colors = {"M": "#88CCEE", "F": "#CC6677"}
subplot_axis = subplot_axes[1]
for sex, color in sex_colors.items():
    sex_mask = analysis_frame["Sex"] == sex
    subplot_axis.scatter(
        analysis_frame.loc[sex_mask, "UMAP1"],
        analysis_frame.loc[sex_mask, "UMAP2"],
        c=color,
        label=sex,
        s=15,
        alpha=0.7,
    )
subplot_axis.set_title("Colored by Sex")
subplot_axis.legend(fontsize=8, markerscale=2)
subplot_axis.set_xticks([])
subplot_axis.set_yticks([])
subplot_axis.set_xlabel("UMAP1")

subplot_axis = subplot_axes[2]
age_scatter = subplot_axis.scatter(
    analysis_frame["UMAP1"],
    analysis_frame["UMAP2"],
    c=analysis_frame["Age"].astype(float),
    cmap="coolwarm",
    s=15,
    alpha=0.7,
)
plt.colorbar(age_scatter, ax=subplot_axis, label="Age")
subplot_axis.set_title("Colored by Age")
subplot_axis.set_xticks([])
subplot_axis.set_yticks([])
subplot_axis.set_xlabel("UMAP1")

plt.suptitle("MAESTRO Latent Embedding UMAP", fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. Diagnosis Classification

Train a logistic regression classifier on the latent embeddings to predict diagnosis, using stratified 5-fold cross-validation.

In [ ]:
features = analysis_frame[feature_columns].values
diagnosis_labels = analysis_frame["Diagnosis"].values

diagnosis_classes = sorted(np.unique(diagnosis_labels))
print(f"Classes: {diagnosis_classes}, N={len(diagnosis_labels)}")

stratified_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
observed_labels, predicted_labels = [], []
fold_accuracies = []

for fold_index, (training_indices, testing_indices) in enumerate(
    stratified_folds.split(features, diagnosis_labels)
):
    feature_scaler = StandardScaler()
    training_features = feature_scaler.fit_transform(features[training_indices])
    testing_features = feature_scaler.transform(features[testing_indices])
    training_labels = diagnosis_labels[training_indices]
    testing_labels = diagnosis_labels[testing_indices]

    classifier = LogisticRegression(max_iter=1000, random_state=42)
    classifier.fit(training_features, training_labels)
    fold_predictions = classifier.predict(testing_features)

    fold_accuracy = accuracy_score(testing_labels, fold_predictions)
    fold_accuracies.append(fold_accuracy)
    observed_labels.extend(testing_labels)
    predicted_labels.extend(fold_predictions)
    print(f"  Fold {fold_index + 1}: Accuracy = {fold_accuracy:.3f}")

print(
    f"\nMean Accuracy: {np.mean(fold_accuracies):.3f} +/- {np.std(fold_accuracies):.3f}"
)

In [ ]:
confusion_counts = confusion_matrix(
    observed_labels, predicted_labels, labels=diagnosis_classes
)
normalized_confusion = confusion_counts.astype(float) / confusion_counts.sum(
    axis=1, keepdims=True
)

_, confusion_axis = plt.subplots(figsize=(5, 4))
sns.heatmap(
    normalized_confusion,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=diagnosis_classes,
    yticklabels=diagnosis_classes,
    ax=confusion_axis,
    vmin=0,
    vmax=1,
    square=True,
)
confusion_axis.set_xlabel("Predicted")
confusion_axis.set_ylabel("True")
confusion_axis.set_title("Diagnosis Classification (Normalized Confusion Matrix)")
plt.tight_layout()
plt.show()

---
## 6. Sex Prediction

Binary classification of sex from latent embeddings.

In [ ]:
sex_frame = analysis_frame[analysis_frame["Sex"].isin(["M", "F"])].copy()
sex_features = sex_frame[feature_columns].values
sex_labels = sex_frame["Sex"].values

stratified_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
observed_sex_labels, predicted_sex_labels = [], []
sex_accuracies = []

for training_indices, testing_indices in stratified_folds.split(
    sex_features, sex_labels
):
    feature_scaler = StandardScaler()
    training_features = feature_scaler.fit_transform(sex_features[training_indices])
    testing_features = feature_scaler.transform(sex_features[testing_indices])

    classifier = LogisticRegression(max_iter=1000, random_state=42)
    classifier.fit(training_features, sex_labels[training_indices])
    fold_predictions = classifier.predict(testing_features)

    fold_accuracy = accuracy_score(sex_labels[testing_indices], fold_predictions)
    sex_accuracies.append(fold_accuracy)
    observed_sex_labels.extend(sex_labels[testing_indices])
    predicted_sex_labels.extend(fold_predictions)

mean_sex_accuracy = np.mean(sex_accuracies)
sex_accuracy_standard_deviation = np.std(sex_accuracies)
print(
    f"Sex Classification — Mean Accuracy: {mean_sex_accuracy:.3f} "
    f"+/- {sex_accuracy_standard_deviation:.3f}"
)

confusion_counts = confusion_matrix(
    observed_sex_labels, predicted_sex_labels, labels=["F", "M"]
)
normalized_confusion = confusion_counts.astype(float) / confusion_counts.sum(
    axis=1, keepdims=True
)

_, confusion_axis = plt.subplots(figsize=(4, 3))
sns.heatmap(
    normalized_confusion,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=["F", "M"],
    yticklabels=["F", "M"],
    ax=confusion_axis,
    vmin=0,
    vmax=1,
    square=True,
)
confusion_axis.set_xlabel("Predicted")
confusion_axis.set_ylabel("True")
confusion_axis.set_title("Sex Prediction")
plt.tight_layout()
plt.show()

---
## 7. Age Regression

Ridge regression to predict age from latent embeddings.

In [ ]:
age_frame = analysis_frame[
    pd.to_numeric(analysis_frame["Age"], errors="coerce").notna()
].copy()
age_frame["Age"] = age_frame["Age"].astype(float)

age_features = age_frame[feature_columns].values
age_labels = age_frame["Age"].values

folds = KFold(n_splits=5, shuffle=True, random_state=42)
observed_ages, predicted_ages = [], []

for training_indices, testing_indices in folds.split(age_features):
    feature_scaler = StandardScaler()
    training_features = feature_scaler.fit_transform(age_features[training_indices])
    testing_features = feature_scaler.transform(age_features[testing_indices])

    regressor = Ridge(alpha=1.0)
    regressor.fit(training_features, age_labels[training_indices])
    fold_predictions = regressor.predict(testing_features)

    observed_ages.extend(age_labels[testing_indices])
    predicted_ages.extend(fold_predictions)

observed_ages = np.array(observed_ages)
predicted_ages = np.array(predicted_ages)
mean_absolute_age_error = mean_absolute_error(observed_ages, predicted_ages)
age_r_squared = r2_score(observed_ages, predicted_ages)

print(
    f"Age Regression — MAE: {mean_absolute_age_error:.2f} years, "
    f"R2: {age_r_squared:.3f}"
)

_, regression_axis = plt.subplots(figsize=(5, 4))
regression_axis.scatter(
    observed_ages,
    predicted_ages,
    alpha=0.6,
    s=20,
    edgecolors="k",
    linewidths=0.5,
)
plot_limits = [
    min(observed_ages.min(), predicted_ages.min()) - 5,
    max(observed_ages.max(), predicted_ages.max()) + 5,
]
regression_axis.plot(
    plot_limits,
    plot_limits,
    "k--",
    alpha=0.5,
    label="Perfect prediction",
)
regression_axis.set_xlabel("True Age")
regression_axis.set_ylabel("Predicted Age")
regression_axis.set_title(
    f"Age Regression (MAE={mean_absolute_age_error:.1f}, R²={age_r_squared:.2f})"
)
regression_axis.legend(fontsize=8)
plt.tight_layout()
plt.show()

---
## 8. Cell Type Proportion Prediction

Predict cell type proportions (computed from the gated labels) using the latent embeddings via Ridge regression.

In [ ]:
proportion_records = []
for sample_name, cell_type_tensor in zip(sample_names, sample_cell_types, strict=True):
    cell_type_names = dataset.get_cell_type_names(cell_type_tensor.numpy())
    cell_type_counts = Counter(cell_type_names)
    total_cells = sum(cell_type_counts.values())
    proportions = {
        cell_type: count / total_cells for cell_type, count in cell_type_counts.items()
    }
    proportions["sample_name"] = sample_name
    proportion_records.append(proportions)

proportion_frame = pd.DataFrame(proportion_records).fillna(0)
proportion_frame = proportion_frame.set_index("sample_name")

proportion_analysis_frame = pd.merge(
    embedding_frame,
    proportion_frame,
    left_on="sample_name",
    right_index=True,
    how="inner",
)

cell_type_columns = [
    column
    for column in proportion_analysis_frame.columns
    if column not in feature_columns and column != "sample_name"
]
print(f"Cell types to predict: {cell_type_columns}")

In [ ]:
proportion_features = proportion_analysis_frame[feature_columns].values
folds = KFold(n_splits=5, shuffle=True, random_state=42)

mean_absolute_errors = {}
for cell_type in cell_type_columns:
    observed_proportions = proportion_analysis_frame[cell_type].values
    fold_observations, fold_predictions = [], []

    for training_indices, testing_indices in folds.split(proportion_features):
        feature_scaler = StandardScaler()
        training_features = feature_scaler.fit_transform(
            proportion_features[training_indices]
        )
        testing_features = feature_scaler.transform(
            proportion_features[testing_indices]
        )

        regressor = Ridge(alpha=1.0)
        regressor.fit(training_features, observed_proportions[training_indices])
        fold_predictions.extend(regressor.predict(testing_features))
        fold_observations.extend(observed_proportions[testing_indices])

    mean_absolute_errors[cell_type] = mean_absolute_error(
        fold_observations, fold_predictions
    )

sorted_cell_types = sorted(mean_absolute_errors, key=mean_absolute_errors.__getitem__)
_, error_axis = plt.subplots(figsize=(8, 4))
error_axis.barh(
    range(len(sorted_cell_types)),
    [mean_absolute_errors[cell_type] for cell_type in sorted_cell_types],
    color="steelblue",
)
error_axis.set_yticks(range(len(sorted_cell_types)))
error_axis.set_yticklabels(sorted_cell_types, fontsize=8)
error_axis.set_xlabel("MAE (proportion)")
error_axis.set_title("Cell Type Proportion Prediction from Latent Embeddings")
plt.tight_layout()
plt.show()

---
## 9. Batch Effect Analysis

Measure embedding stability by applying random marker intensity shifts and comparing the resulting latent representations.

In [ ]:
assert model is not None, "Model not loaded."


def apply_marker_shift(
    sample_tensor: torch.Tensor,
    gamma_shape: float = 1.5,
    gamma_scale: float = 3.0,
) -> torch.Tensor:
    """Apply random marker-specific intensity shifts."""
    shifted_input = sample_tensor.float()
    _, _, marker_count = shifted_input.shape
    shifts = (
        torch.from_numpy(
            np.random.gamma(gamma_shape, gamma_scale, size=(1, 1, marker_count))
        )
        .float()
        .to(shifted_input.device)
    )
    signs = (
        torch.from_numpy(np.random.choice([-1, 1], size=(1, 1, marker_count)))
        .float()
        .to(shifted_input.device)
    )
    return shifted_input + shifts * signs


shifted_embeddings = {}
np.random.seed(42)

for sample_index in tqdm(
    range(len(sample_tensors)), desc="Computing shifted embeddings"
):
    with torch.no_grad():
        sample_tensor = sample_tensors[sample_index].to(device)
        shifted_input = apply_marker_shift(sample_tensor)
        encoded_cells, *_ = model.student.forward_encoder(shifted_input)
        shifted_embedding, _, _ = model.student.forward_pooling(encoded_cells)
        shifted_embeddings[sample_names[sample_index]] = shifted_embedding.cpu()

sorted_sample_names = sorted(sample_embeddings)
embedding_distances = [
    np.linalg.norm(
        sample_embeddings[sample_name].flatten().numpy()
        - shifted_embeddings[sample_name].flatten().numpy()
    )
    for sample_name in sorted_sample_names
]

print(f"Mean L2 distance (original vs shifted): {np.mean(embedding_distances):.4f}")
print(f"Standard deviation of L2 distance: {np.std(embedding_distances):.4f}")

_, distance_axis = plt.subplots(figsize=(5, 3))
distance_axis.hist(embedding_distances, bins=20, color="steelblue", edgecolor="white")
distance_axis.axvline(
    np.mean(embedding_distances),
    color="red",
    linestyle="--",
    label=f"Mean = {np.mean(embedding_distances):.3f}",
)
distance_axis.set_xlabel("L2 Distance")
distance_axis.set_ylabel("Count")
distance_axis.set_title("Embedding Stability Under Marker Shifts")
distance_axis.legend(fontsize=8)
plt.tight_layout()
plt.show()

---
## 10. Pool Attention Interpretation

Analyze which cell types receive the most attention during pooling, revealing what the model considers most informative.

In [ ]:
assert model is not None, "Model not loaded."

attention_sample_names = []
attention_results = []

for sample_index in tqdm(range(len(sample_tensors)), desc="Extracting pool attention"):
    with torch.no_grad():
        sample_tensor = sample_tensors[sample_index].to(device)
        cell_types = sample_cell_types[sample_index]

        encoded_cells, *_ = model.student.forward_encoder(sample_tensor)
        _, _, pooling_attention = model.student.forward_pooling(encoded_cells)

        # Pooling attention has shape [heads, seeds, cells].
        attention_weights = pooling_attention.mean(dim=0).squeeze(0).cpu().numpy()

        cell_type_names = dataset.get_cell_type_names(cell_types)

        attention_by_cell_type = {}
        for cell_type, attention_weight in zip(
            cell_type_names, attention_weights, strict=True
        ):
            if cell_type not in attention_by_cell_type:
                attention_by_cell_type[cell_type] = []
            attention_by_cell_type[cell_type].append(attention_weight)

        mean_attention_by_cell_type = {
            cell_type: np.mean(weights)
            for cell_type, weights in attention_by_cell_type.items()
        }
        attention_results.append(mean_attention_by_cell_type)
        attention_sample_names.append(sample_names[sample_index])

all_cell_types = sorted(
    {
        cell_type
        for attention_result in attention_results
        for cell_type in attention_result
    }
)
attention_records = []
for sample_index, sample_name in enumerate(attention_sample_names):
    record = {"sample_name": sample_name}
    for cell_type in all_cell_types:
        record[cell_type] = attention_results[sample_index].get(cell_type, 0.0)
    attention_records.append(record)

attention_frame = pd.DataFrame(attention_records)
print(f"Attention DataFrame shape: {attention_frame.shape}")
attention_frame.head()

In [ ]:
mean_attention = attention_frame[all_cell_types].mean().sort_values(ascending=True)

_, attention_axis = plt.subplots(figsize=(7, 4))
bar_colors = [
    cell_type_colors.get(cell_type, default_color) for cell_type in mean_attention.index
]
attention_axis.barh(range(len(mean_attention)), mean_attention.values, color=bar_colors)
attention_axis.set_yticks(range(len(mean_attention)))
attention_axis.set_yticklabels(mean_attention.index, fontsize=8)
attention_axis.set_xlabel("Mean Pool Attention Weight")
attention_axis.set_title("Pool Attention by Cell Type")
plt.tight_layout()
plt.show()

In [ ]:
attention_with_metadata = pd.merge(
    attention_frame, metadata, on="sample_name", how="inner"
)

_, subplot_axes = plt.subplots(
    1, len(diagnosis_colors), figsize=(5 * len(diagnosis_colors), 4), sharey=True
)

for subplot_axis, diagnosis in zip(subplot_axes, diagnosis_colors, strict=True):
    diagnosis_attention = attention_with_metadata[
        attention_with_metadata["Diagnosis"] == diagnosis
    ]
    mean_values = diagnosis_attention[all_cell_types].mean().sort_values(ascending=True)
    bar_colors = [
        cell_type_colors.get(cell_type, default_color)
        for cell_type in mean_values.index
    ]
    subplot_axis.barh(range(len(mean_values)), mean_values.values, color=bar_colors)
    subplot_axis.set_yticks(range(len(mean_values)))
    subplot_axis.set_yticklabels(mean_values.index, fontsize=7)
    subplot_axis.set_title(f"{diagnosis} (n={len(diagnosis_attention)})", fontsize=10)
    subplot_axis.set_xlabel("Mean Attention")

plt.suptitle("Pool Attention Stratified by Diagnosis", fontsize=13)
plt.tight_layout()
plt.show()

---
## Summary

This notebook demonstrated the key downstream analyses enabled by MAESTRO:

- **Reconstruction**: The model can reconstruct masked cells, preserving cell type structure.
- **Latent embeddings**: Each patient is compressed to a fixed-size vector capturing immune state.
- **Classification/Regression**: Latent embeddings support diagnosis, sex, and age prediction.
- **Cell type proportions**: The model implicitly learns cell type composition.
- **Batch robustness**: Embeddings are stable under simulated batch effects.
- **Attention**: Pool attention reveals which cell types are most informative.

For your own analyses, replace the toy data with your CyTOF datasets and update the model checkpoint paths.